# 🦜🔗 LangChain Agents — Demo Notebook
### Creating Agents with Custom Tools using LangChain v1.0.0
> **Python 3.11 | LangChain 1.0.0 | Custom LLM (Capgemini API)**


In [ ]:
%pip install langchain==1.0.0 langchain-core langgraph langgraph-prebuilt langchain-openai -q
print("✅ All packages installed successfully!")

## 📦 Step 1: Imports

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

print("✅ All imports successful!")

## 🔧 Step 2: Configure the LLM (Capgemini API)
> Replace the placeholders below with your actual Capgemini API credentials.

In [ ]:
# ─── Replace these with your actual Capgemini API details ───
CAPGEMINI_API_KEY   = "YOUR_API_KEY_HERE"
CAPGEMINI_BASE_URL  = "https://your-capgemini-base-url.com"   # e.g. https://api.capgemini.com/openai/v1
CAPGEMINI_MODEL     = "gpt-4o"                                 # or the model name provided to you
CAPGEMINI_API_VER   = "2024-02-01"                             # Azure API version if applicable
# ─────────────────────────────────────────────────────────────

llm = AzureChatOpenAI(
    azure_endpoint=CAPGEMINI_BASE_URL,
    api_key=CAPGEMINI_API_KEY,
    azure_deployment=CAPGEMINI_MODEL,
    api_version=CAPGEMINI_API_VER,
    temperature=0,
)

print(f"✅ LLM configured: {CAPGEMINI_MODEL}")
print(f"   Endpoint: {CAPGEMINI_BASE_URL}")

## 🛠️ Step 3: Define Custom Tools
We define tools using the `@tool` decorator from `langchain_core.tools`.  
The **docstring** acts as the tool description — the agent reads this to decide when to use each tool.

In [ ]:
@tool
def get_employee_info(employee_id: str) -> str:
    """
    Fetches employee information from the HR system.
    Use this tool when the user asks about an employee's details such as
    name, department, role, or location given an employee ID.
    """
    # Dummy data — simulating an HR database response
    dummy_db = {
        "E001": {"name": "Alice Johnson",   "dept": "Engineering",  "role": "Senior Dev",   "location": "Paris"},
        "E002": {"name": "Bob Smith",       "dept": "Marketing",    "role": "Campaign Lead","location": "London"},
        "E003": {"name": "Priya Sharma",    "dept": "Data Science", "role": "ML Engineer",  "location": "Mumbai"},
    }
    if employee_id in dummy_db:
        info = dummy_db[employee_id]
        return (f"Employee ID : {employee_id}\n"
                f"Name        : {info['name']}\n"
                f"Department  : {info['dept']}\n"
                f"Role        : {info['role']}\n"
                f"Location    : {info['location']}")
    return f"❌ No employee found with ID '{employee_id}'."


@tool
def get_project_status(project_name: str) -> str:
    """
    Returns the current status of an internal project.
    Use this when the user asks about the progress, deadline, or status
    of a specific project by name.
    """
    dummy_projects = {
        "Apollo":  {"status": "In Progress", "deadline": "2025-06-30", "team_size": 8},
        "Orion":   {"status": "Completed",   "deadline": "2025-03-15", "team_size": 5},
        "Horizon": {"status": "Planning",    "deadline": "2025-09-01", "team_size": 12},
    }
    name = project_name.strip().title()
    if name in dummy_projects:
        p = dummy_projects[name]
        return (f"Project     : {name}\n"
                f"Status      : {p['status']}\n"
                f"Deadline    : {p['deadline']}\n"
                f"Team Size   : {p['team_size']} members")
    return f"❌ No project found with name '{project_name}'."


tools = [get_employee_info, get_project_status]

print(f"✅ {len(tools)} tools registered:")
for t in tools:
    print(f"   🔧 {t.name} — {t.description[:60]}...")

## 🤖 Step 4: Create the Agent
We use `create_agent` from `langchain.agents` (LangChain v1.0.0).  
The agent automatically decides **which tool to call** and **when to stop** based on the user query.

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful internal assistant for Capgemini employees. "
        "You have access to HR and project management tools. "
        "Always use the available tools to answer questions accurately. "
        "Be concise and professional."
    ),
)

print("✅ Agent created successfully!")
print(f"   Tools available to agent: {[t.name for t in tools]}")

## ▶️ Step 5: Run the Agent — Employee Query
Let's ask the agent about an employee. It should automatically call `get_employee_info`.

In [ ]:
query_1 = "Can you tell me about employee E003?"

print(f"🧑 User: {query_1}")
print("-" * 50)

response_1 = agent.invoke({
    "messages": [HumanMessage(content=query_1)]
})

# Extract the final text response
final_answer_1 = response_1["messages"][-1].content
print(f"🤖 Agent: {final_answer_1}")

## ▶️ Step 6: Run the Agent — Project Status Query
Now let's ask about a project. The agent should call `get_project_status`.

In [ ]:
query_2 = "What is the current status of Project Apollo?"

print(f"🧑 User: {query_2}")
print("-" * 50)

response_2 = agent.invoke({
    "messages": [HumanMessage(content=query_2)]
})

final_answer_2 = response_2["messages"][-1].content
print(f"🤖 Agent: {final_answer_2}")

## ▶️ Step 7: Multi-Tool Query
Ask something that requires using **both tools** or chaining reasoning.

In [ ]:
query_3 = "Who is employee E001 and are they working on Project Horizon?"

print(f"🧑 User: {query_3}")
print("-" * 50)

response_3 = agent.invoke({
    "messages": [HumanMessage(content=query_3)]
})

final_answer_3 = response_3["messages"][-1].content
print(f"🤖 Agent: {final_answer_3}")

## 🔍 Step 8: Inspect Tool Calls (Under the Hood)
Let's look at all the messages in the agent's response to see which tools were called.

In [ ]:
print("📋 Full message trace for Query 3:\n")
for i, msg in enumerate(response_3["messages"]):
    msg_type = type(msg).__name__
    print(f"[{i}] {msg_type}")
    if hasattr(msg, "content") and msg.content:
        print(f"    Content : {msg.content[:120]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"    ToolCall: {tc['name']}({tc['args']})")
    print()

## ✅ Summary

| Concept | What We Did |
|---|---|
| `@tool` decorator | Defined custom tools with docstring descriptions |
| `create_agent()` | Created a LangChain v1.0.0 agent with LLM + tools |
| Agent invocation | Called `agent.invoke()` with a user message |
| Tool routing | Agent automatically decided which tool to call |
| Message trace | Inspected intermediate tool calls in the response |

### 🔑 Key Imports
```python
from langchain.agents import create_agent        # LangChain v1.0.0
from langchain_core.tools import tool            # Tool decorator
from langchain_openai import AzureChatOpenAI     # LLM (swap for Capgemini)
```

> 💡 **Note:** Replace `AzureChatOpenAI` with the appropriate LangChain chat model class that matches your Capgemini API endpoint format.
